In [1]:
## READS MNQ-TICK from HGT-Export SC chartbook

In [2]:
import pandas as pd
import numpy as np
import datetime as dt 
from pathlib import Path
import time

In [3]:
# Configuration: Style preferences
#plt.style.use('ggplot') # Good default for readability
pd.set_option("display.width", 400)      # total characters per line
pd.set_option("display.max_columns", 30) # prevent wrapping by limiting columns
pd.set_option("display.max_rows", 1000)

In [4]:
import os
os.getcwd()

'/home/vm/pt/hgt-rl/mnq-tick/iteration3a/all-raw'

In [5]:
#symbol = 'mnq'
#SEC = 2

#inFile = f'/mnt/d/SierraChart/data/EXPORT/MNQ-TICK-OSCILLATOR-6SEC.csv'
#outFile= f'data/mnq-tick-oscillator-6sec.pqt'
inFile = f'/mnt/d/SierraChart/data/EXPORT/MNQ-ALL-RAW-3SEC.csv'
outFile= f'data/mnq-tick-all-raw-3sec.pqt'

print(inFile, outFile)

/mnt/d/SierraChart/data/EXPORT/MNQ-ALL-RAW-3SEC.csv data/mnq-tick-all-raw-3sec.pqt


In [6]:
df = pd.read_csv(inFile)

print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 11117814 entries, 0 to 11117813
Data columns (total 34 columns):
 #   Column         Dtype  
---  ------         -----  
 0   Date           str    
 1   Time           str    
 2   Open           float64
 3   High           float64
 4   Low            float64
 5   Last           float64
 6   Volume         int64  
 7   #ofTrades      int64  
 8   OHLCAvg        float64
 9   HLCAvg         float64
 10  HLAvg          float64
 11  BidVolume      int64  
 12  AskVolume      int64  
 13  JMA            float64
 14  VEL            float64
 15  ZeroLevel      float64
 16  VEL.1          float64
 17  ZeroLevel.1    float64
 18  RSX            float64
 19  TopLevel       float64
 20  BottomLevel    float64
 21  ZeroLevel.2    float64
 22  RSX.1          float64
 23  TopLevel.1     float64
 24  BottomLevel.1  float64
 25  ZeroLevel.3    float64
 26  Momentum       float64
 27  Line           float64
 28  Momentum.1     float64
 29  Line.1         float64


In [7]:
# expand stupid SC date like 2026-7-8 to 2026-07-08
parts = df['Date'].astype(str).str.split('-', expand=True)

year  = parts[0]
month = parts[1].str.zfill(2)
day   = parts[2].str.zfill(2)

df['Date_norm'] = year + '-' + month + '-' + day

# join date and time and convert to wall clock datetime64
df['timestamp'] = pd.to_datetime(
    df['Date_norm'] + ' ' + df['Time'].astype(str),
    utc=False,            # keep as wall clock, no timezone
    errors='raise'        # or 'coerce' if you want bad rows as NaT
)

# remove temp columns
df = df.drop(columns=['Date', 'Date_norm', 'Time'])

# make timestamp first column
col = df.pop('timestamp')
df.insert(0, 'timestamp', col)

print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 11117814 entries, 0 to 11117813
Data columns (total 33 columns):
 #   Column         Dtype         
---  ------         -----         
 0   timestamp      datetime64[us]
 1   Open           float64       
 2   High           float64       
 3   Low            float64       
 4   Last           float64       
 5   Volume         int64         
 6   #ofTrades      int64         
 7   OHLCAvg        float64       
 8   HLCAvg         float64       
 9   HLAvg          float64       
 10  BidVolume      int64         
 11  AskVolume      int64         
 12  JMA            float64       
 13  VEL            float64       
 14  ZeroLevel      float64       
 15  VEL.1          float64       
 16  ZeroLevel.1    float64       
 17  RSX            float64       
 18  TopLevel       float64       
 19  BottomLevel    float64       
 20  ZeroLevel.2    float64       
 21  RSX.1          float64       
 22  TopLevel.1     float64       
 23  BottomLevel.1  f

In [8]:
'''

'''

df.drop(columns=['Volume','OHLCAvg','HLCAvg','HLAvg','BidVolume','AskVolume', '#ofTrades',
                 'ZeroLevel','ZeroLevel.1','TopLevel','BottomLevel','ZeroLevel.2','TopLevel.1','BottomLevel.1','ZeroLevel.3','Line','Line.1','Line.2'], inplace=True) 

df.rename(columns={
    'Open': 'rawOpen',
    'High': 'rawHigh',
    'Low' : 'rawLow',
    'Last': 'rawLast',
    
    'VEL.1': 'adpVEL',
    'TopBand': 'bolTopAdpVEL',
    'MiddleBand': 'bolMidAdpVEL',
    'BottomBand': 'bolBotAdpVEL',    
    'RSX.1': 'tickRSX',
    'Momentum': 'jmaD1',
    'Momentum.1': 'jmaD2',
    'TopBand.1': 'bolTopJmaD2',
    'MiddleBand.1': 'bolMidJmaD2',
    'BottomBand.1': 'bolBotJmaD2',
    'Momentum.2': 'tickJmaD1',
    'Momentum.3': 'tickJmaD2',
    'TopBand.2': 'bolTopTickJmaD2',
    'MiddleBand.2': 'bolMidTickJmaD2',
    'BottomBand.2': 'bolBotTickJmaD2',
    'JMA.1': 'tickJMA',
}, inplace=True)

print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 11117814 entries, 0 to 11117813
Data columns (total 15 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[us]
 1   rawOpen    float64       
 2   rawHigh    float64       
 3   rawLow     float64       
 4   rawLast    float64       
 5   JMA        float64       
 6   VEL        float64       
 7   adpVEL     float64       
 8   RSX        float64       
 9   tickRSX    float64       
 10  jmaD1      float64       
 11  jmaD2      float64       
 12  tickJmaD1  float64       
 13  tickJmaD2  float64       
 14  tickJMA    float64       
dtypes: datetime64[us](1), float64(14)
memory usage: 1.2 GB
None
            timestamp  rawOpen   rawHigh    rawLow   rawLast           JMA        VEL  adpVEL        RSX    tickRSX      jmaD1      jmaD2   tickJmaD1  tickJmaD2     tickJMA
0 2022-01-03 08:00:00  16431.0  16431.00  16429.75  16430.25  16333.539062   0.956565     0.0  42.986149  45.147453   0.000000   0.

In [9]:
print(f'monotonic: {df["timestamp"].is_monotonic_increasing}')

df['date'] = df['timestamp'].dt.normalize()

filtered_days = [
   '2022-01-17', '2022-07-04', '2022-11-24', '2023-01-16',
   '2023-02-20', '2023-04-07', '2023-05-29', '2023-06-19',
   '2023-07-04', '2023-09-04', '2023-11-23', '2024-02-19',
   '2024-05-27', '2024-06-19', '2024-07-04', '2024-11-28',
   '2025-01-09', '2025-07-04', '2025-09-01', '2025-11-27',
   '2026-04-03', '2026-07-09',               '2025-11-28'
]

# before 1168
days = df["date"]
all_days = days.drop_duplicates().sort_values()
print("Unique days before:", len(all_days))

df = df[
    ~df["timestamp"].dt.normalize().isin(pd.to_datetime(filtered_days))
]

# after -23 = 1145
days = df["date"]
all_days = days.drop_duplicates().sort_values()
print(" Unique days after:", len(all_days))

#

print(df.info())
print(df.head())

monotonic: True
Unique days before: 1179
 Unique days after: 1156
<class 'pandas.DataFrame'>
Index: 11003371 entries, 0 to 11117813
Data columns (total 16 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[us]
 1   rawOpen    float64       
 2   rawHigh    float64       
 3   rawLow     float64       
 4   rawLast    float64       
 5   JMA        float64       
 6   VEL        float64       
 7   adpVEL     float64       
 8   RSX        float64       
 9   tickRSX    float64       
 10  jmaD1      float64       
 11  jmaD2      float64       
 12  tickJmaD1  float64       
 13  tickJmaD2  float64       
 14  tickJMA    float64       
 15  date       datetime64[us]
dtypes: datetime64[us](2), float64(14)
memory usage: 1.4 GB
None
            timestamp  rawOpen   rawHigh    rawLow   rawLast           JMA        VEL  adpVEL        RSX    tickRSX      jmaD1      jmaD2   tickJmaD1  tickJmaD2     tickJMA       date
0 2022-01-03 08:00:00  16431

In [10]:
df.to_parquet(outFile, index=False)
print(f'written to: {outFile}')

written to: data/mnq-tick-all-raw-3sec.pqt


In [11]:
print(df["timestamp"].dt.time.min())    
print(f'dates range: {df["date"].min()} .. {df["date"].max()}')    
x = 1/0

08:00:00
dates range: 2022-01-03 00:00:00 .. 2026-07-24 00:00:00


ZeroDivisionError: division by zero

In [ ]:
# find out why one is larger than the other by 1 day
outFile1= f'data/mnq-tick-full-3sec.pqt'
outFile2= f'data/mnq-ohlc-raw-3sec.pqt'

df1 = pd.read_parquet(outFile1)
df2 = pd.read_parquet(outFile2)

days1 = df1['date'].unique()
days2 = df2['date'].unique()

print(len(days1), len(days2))

missing = [x for x in days2 if x not in days1]
print(missing)

In [ ]:
## WHY 3STREAM DONT MATCH BETWEEN OLD AND NEW
fsrc1 = f'data/mnq-tick-all-3sec.pqt'
fsrc2 = f'data/mnq-tick-full-3sec.pqt'
fraw = f'data/mnq-ohlc-raw-3sec.pqt'

src1 = pd.read_parquet(fsrc1)
src2 = pd.read_parquet(fsrc2)
raw  = pd.read_parquet(fraw)


In [ ]:
pd.set_option('display.float_format', lambda x: f'{x:.8f}')

print(src1.head(100))
display(src2.head())
display(raw.head())